In [ ]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

library(dplyr)
library(WGCNA)
library(data.table)

source("/mnt/lareaulab/reliscu/code/FindModules/FindModules.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")

Allowing multi-threading with up to 48 threads.


In [ ]:
data_source <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_Claude_marker_genes_Micro_PVM_Oligo_Astro_VLMC_TopModPosBC_genes_removed"

expr <- fread(paste0("data/", data_source, ".csv"), data.table=FALSE)

colnames(expr)[1] <- "Gene"

## Run FM

In [ ]:
# Subset to genes in the top X percentile for generating modules

prob <- .6
mean_expr <- rowMeans(expr[,-1])

subset_cutoff <- unname(quantile(mean_expr, prob))
print(paste("quantile(mean_expr, prob):", round(subset_cutoff, 3)))
subset <- mean_expr >= subset_cutoff
sum(subset)

In [ ]:
# Order samples by covariates of interest

sampleinfo <- read.csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv")

sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[sampleinfo[,1] %in% colnames(expr),]

sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))
sampleinfo$SAMPID <- make.names(sampleinfo$SAMPID)

sampleinfo <- sampleinfo %>% arrange(Mean_age)
    
expr <- expr[, c(1, match(sampleinfo[,1], colnames(expr)[-1]) + 1)]
all.equal(sampleinfo[,1], colnames(expr)[-1])

In [ ]:
samplegroups <- as.factor(sampleinfo$AGE)
merge.param <- 0.95
projectname <- paste0(data_source, "_mergeParam", merge.param, "_subsetCutoff", round(subset_cutoff, 3))

In [ ]:
FindModules(
  projectname=projectname,
  expr=expr,
  geneinfo=1,
  sampleindex=2:ncol(expr),
  samplegroups=samplegroups,
  subset=subset,
  simMat=NULL,
  saveSimMat=FALSE,
  simType="Bicor",
  beta=1,
  overlapType="None",
  TOtype="signed",
  TOdenom="min",
  MIestimator="mi.mm",
  MIdisc="equalfreq",
  signumType="rel",
  iterate=TRUE,
  signumvec=c(.99999,.9999,.999,.99,.98,.97,.96), 
  minsizevec=c(3,4,6,8,10), 
  signum=NULL,
  minSize=NULL,
  merge.by="ME",
  merge.param=merge.param,
  export.merge.comp=T,
  ZNCcut=2,
  calcSW=FALSE,
  loadTree=FALSE,
  writeKME=TRUE,
  calcBigModStat=FALSE,
  writeModSnap=TRUE
)

## Get enrichments

## Filter genes

In [ ]:
# Filter seed genes in noise modules

In [ ]:
network_dir <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_residuals_subtracted_mergeParam0.95_subsetCutoff0.001_Modules"

In [ ]:
# Remove genes in modules with specific enrichments

top_mods_df <- fread("data/enrichments/", data.table=FALSE)


In [ ]:
top_mods_df %>%
    arrange(Qval) %>%
    select(Cell_type, Qval) %>%
    head(n=10)

In [ ]:
sets <- c("Micro_PVM", "Oligo", "Astro", "VLMC")

top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets,]

In [ ]:
mod_genes_list <- lapply(seq_along(1:nrow(top_mods_df_subset)), function(i) {
        kME_path <- top_mods_df$kME_path[i]
        module <- top_mods_df$Module[i]
        get_mod_genes(kME_path, module, mod_def, n_genes=NULL)
    })
mod_genes <- unlist(mod_genes_list)